# Tutoriel "Quickstart" de Pytorch

In [1]:
# On commence par importer les différents modules Pytorch que l'on utilisera
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# Chargement de la base de données

In [2]:
# On télécharge les données d'entrainement depuis une base de données libre, ici FashionMNIST
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# On télécharge les données de test de la même base de données.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100.0%
100.0%
100.0%
100.0%


## La base de données "FashionMNIST"

La base de données **FashionMNIST** est nommée ainsi en référence à la base de données très connue nommé **MNIST** qui contient différentes écritures de chiffres.
Plutôt que des chiffres, cette base de données contient des **images d'articles de modes** du site *Zalando*.

Pour obtenir plus d'informations: [PyTorch docs](https://pytorch.org/vision/stable/generated/torchvision.datasets.FashionMNIST.html)  
Leur github officiel: [zalandoresearch/fashion-mnist](https://github.com/zalandoresearch/fashion-mnist)

Cette base de données contient:
- **60,000 images d'entrainement**
- **10,000 images de test**

Chaque image est un **niveau de gris de 28x28 pixels** qui appartient à une des **1O classes** possibles (T-shirt, pantalon, pull, ...)

On peut observer dans le code au desus qu'il y a un paramêtre booléen `train` dans le module `torchvision.datasets.FashionMNIST` qui indique si on veut les images d'entrainement ou celles de test.

Paramètres importants:
- `download=True` → Télécharge localement la base de données si elle n'est pas déjà présente.  
- `transform=ToTensor()` → convertis les images ou format PIL en des tenseurs Pytorch pour qu'ils puissent être utilisés par le modèle

In [3]:
batch_size = 64

# Créé les "dataloaders".
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## Les "DataLoader"

Les classes **DataLoader** sont utilisées pour charger les données d'une base de données Pytorch et d'itérer dessus efficacement.

Elles prennent principalement deux paramètres:
- La **base de données** (e.g. `train_dataset`)  
- Une **taille d'échantillonnage** ou *batch size* soit le nombre d'élément à envoyer à notre modèle par passage aller-retour (les "forward/backward pass").

Une taille d'échantillonnage élevée utilise plus de **mémoire RAM ou VRAM** mais peut amélirer la stabilité lors de l'entraînement.

Les données sont habituellement représentées dans le **format NCHW** qui représente la forme d'une tenseur d'image:

- **N** → nombre d'éléments dans le batch  
- **C** → nombre de cannaux (pour une image en RVB on a C = 3 et pour une image en niveaux de gris on a C = 1)  
- **H** → hauteur de l'image  
- **W** → largeur de l'image

Ces quatres axes définissent les dimensions de tenseur d'image qu'utilise Pytorch.

Plus d'informations sur la [documentation du DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)


# II- Créer un modèle

In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"


print(f"Using {device} device")

# Définis le modèle
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Pour créer un réseau de neurones en Pytorch, nous **devons** créer une classe qui hérite de `nn.Module`.
La documentation de `nn.Module` se trouve [ici](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html).

Nous devons également choisir entre utiliser un accélérateur come **CUDA** qui utilise la carte graphique ou rester sur l'utilisation du **processeur** ou **CPU**.


### Définir notre classe de réseau de neurones

Dans notre constructeur, on définit un premier attribut qui est un `nn.Flatten()`.

Cette couche d'aplatissement convertit chaque image de taille 20x28 en une liste contigue de taille 784, ce qui repésente les pixels de l'image. C'est un passage d'une échelle 2D à une échelle 1D.

Une liste contigue est une liste stockée dans un **bloc de mémoire continu et insécable**.
On peut retrouver à [ce lien](https://stackoverflow.com/questions/26998223/what-is-the-difference-between-contiguous-and-non-contiguous-arrays) une explication illustrée.

Nous définssons ensuite un attribut **`nn.Sequential`** que nous appelons `self.linear_relu_stack`.

Une couche séquentielle est un élément permettant aux données de passer de façon séquentielle à travers de multiples couches. Dans notres cas, cela correspond à un mixte entre des couches de transformations linéaires et ReLU.

Plus d'informations sur le module séquentiel de Pytorch sont disponibles à [ce lien](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html#torch.nn.Sequential).



### Les couches à l'intérieur du `nn.Sequential`

* **`nn.Linear`**: Une couche appliquant une transformation linéaire de la forme $y = Ax + b$ sur les données d'entrées en utilisant le poids $A$ et le biais $b$ propre à chaque couche linéaire.
* **`nn.ReLU`**: Une couche d'activation non-linéaire qui applique $y=0$ si $x \le 0$ et renvoie sinon $x=y$. C'est à dire que cette couche filtre les valeurs qui lui sont passées en entrée pour ne garder que les valeurs positives. Pour plus d'informations, allez [ici](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html).



### Le constructeur et la méthode de passe en avant

Le constructeur `__init__` n'est appellé **qu'une fois** et c'est à la création du réseau de neurones.
At ce moment, toutes les couches (aussi bien celle d'aplatissement que la séquentielle et les linéaires et ReLU en son seins) sont seulement définies, elles n'ont pas encore été appliquées à des données pour l'instant.


La méthode de passe en avant **`forward(self, x)`** est appellée à **chaque fois** que des données passent à travers le réseau de neurones.

* La donnés d'entrée `x` est d'abord **aplatie**.
* Elle passe ensuite à travers la couche `self.linear_relu_stack` et donc les couches de transformations linéaires et ReLU.

Le résultat de cette méthode est appellé **logits** ce qui correspond à un **résultat non-normalisé** du modèle.
Souvent on utilise ensuite la fonction **softmax** pour normaliser.

Les logits sont les prédictions de notre modèle, ils ont besoins d'être comparés à la valeur recherchée afin d'améliorer notre modèle.

Avec les logits, on peut ainsi  définir la **fonction de perte** ou **loss function** et appliquer la **backpropagation** pour entraîner les paramètres de notre modèle (les biais et les poids).

# III - Entraîner un modèle

In [5]:
loss_fn = nn.CrossEntropyLoss()

Pour entraîner notre modèle, nous devons définir sa fonction de perte ou **loss function** et son optimiseur ou **optimizer**.

Dans le cas présent, on utilise la méthode de l'entropie croisée ou **Cross Entropy** pour la fonction de perte. C'est une des fonctions les plus communément utilisées parmis les différentes [fonctions de perte disponibles avec Pytorch](https://pytorch.org/docs/stable/nn.html#loss-functions).


## Une ~~rapide~~ explication des fonctions de perte les plus populaires.

Pour toutes les formules qui suivent, on a:
- $y_i$ = valeur cible, celle que l'on aimerait atteindre
- $\hat{y}_i$ = valeur prédite, obtenue
- $N$ = nombre d'éléments


### 1 - L'erreur quadratique moyenne ou Mean Squared Error (MSELoss)

Utilisée pour des **tâches de régression**, lorsque l'on souhaite prédire des valeurs continues telles que  la température, la bourse, etc...

$$
\mathrm{MSE}(y, \hat{y}) = \frac{1}{N} \sum_{i=1}^{N} \bigl(y_i - \hat{y}_i\bigr)^2
$$

MSE pénalise énormément les grandes érreurs parce que l'on prend en compte le **carré** de la différence entre la prédiction et la valeur cible.
![MSE](https://miro.medium.com/v2/resize:fit:640/format:webp/1*WfVDoLsarrM5HpO9sh_ZQQ.png)


### 2 - L'erreur absolue moyenne ou Mean Absolute Error (L1Loss / MAE)

Utilisée pour des **tâches de régression** si l'on souhaite une résistance aux **valeurs abérrantes**, on veut que ces dernières aient un impact moindre.
$$
\mathrm{MAE}(y, \hat{y}) = \frac{1}{N} \sum_{i=1}^{N} \bigl|y_i - \hat{y}_i\bigr|
$$
Contaîrement à la MSE, on applique une **pénalité linéaire** plutôt qu'une carrée.
Le carré donne beaucoup de poids aux valeurs abérrantes trop élevées, utiliser une pénalité linéaire réduit donc l'impact de ces valeurs abérrantes mais le paysage d'optimisation (optimization landscape) devient moins lisse, ce qui peut produire des gradients très faibles et ainsi compliquer l’optimisation.
![L1_loss](https://miro.medium.com/v2/resize:fit:640/format:webp/1*0hbNOtpfr6aoR_Bmty-JkA.jpeg)



### 3 - L'entropie croisée ou Cross Entropy Loss

L'entropie croiséee est utilisée pour les problèmes de **classification multi-classes**.

Pour chaque valeur d'entrée, le modèle produit un ensemble de **scores ou logits**, un par classe.
Ces scores sont ensuite convertis en probabilités dont la somme fait 1 avec la fonction **Softmax**. (ex: 0.3 pour le label "chien" donne une probabilité de 30% que l'entrée corresponde à la classe "chien").

La "loss" compare les probabilités prédites à celles de la "bonne réponse". La distribution de cette "bonne réponse" est une **distribution parfaite**, la probabilité de la bonne classe est 1 est celle des autres est de 0.

$$
\mathrm{CrossEntropy}(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^{N} \log\left( \frac{e^{\hat{y}_{i, y_i}}}{\sum_{j} e^{\hat{y}_{i,j}}} \right)
$$

#### Explications:

1. Applique la fonction **Softmax** pour convertir les *logits* en probabilités:

$$
p_{i,j} = \frac{e^{\hat{y}_{i,j}}}{\sum_{k} e^{\hat{y}_{i,k}}}
$$

2. Calcul ensuite la moyenne de log de vraissemblance négative ou **average negative log-likelihood** de la classe juste, celle que l'on cherche à obtenir:

$$
\mathrm{CrossEntropy}(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^{N} \log\left(p_{i, y_i}\right)
$$

Ça encourage le modèle à donner une valeur de probabilité plus élevée à la bonne classe.

![cross_entropy](https://ml-cheatsheet.readthedocs.io/en/latest/_images/cross_entropy.png)


### 4 - L'entropie croisée binaire ou Binary Cross-Entropy (BCE)

Utilisée pour des **classifications binaires** (ex: pouvoir dire si un mail est un spam ou ne l'est pas).

$$
\mathrm{BCE}(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \right]
$$

Ici:
- La valeur cible est $y_i$ ∈ {0, 1}  
- La prédiction $\hat{y}_i \in [0, 1]$ représente la **probabilité** d'appartenir à la classe 1 (ex: en notant "spam" = 1 et "pas spam" = 0).

Pour obtenir  $\hat{y}_i$, on applique une fonction  **sigmoïde** au résultat obtenu en passant les données au modèle, aux *logits*:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

Si tu veux directement avoir les *logits* normalisés (donc ne pas avoir à appliquer la fonction sigmoïde) utilise `nn.BCEWithLogitsLoss`, cette méthode combine les deux opérations (la loss et la fonction sigmoïde) de façon fiable.

![binary_Xentropy](https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2F54c97fda8af4dccc23d58bd14cd95802df6f1e49-393x272.png&w=640&q=75)



### 5 - La vraisemblance négative logarthmique ou Negative Log Likelihood Loss (NLLLoss)

Utilisée pour des tâches de **classification multi-classes** lorsque le modèle renvoie directement une **probabilité logarithmique** plutôt que des logits non-normalisés.

$$
\mathrm{NLL}(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^{N} \log p_{i, y_i}
$$

C'est mathématiquement équivalent à **l'entropie croiséee** mais attends d'avoir seulement en entrée des données qui ont déjà été passées par la fonction Softmax.

En Pytorch, c'est fait en utilisant la fonction `nn.LogSoftmax`:

$$
\log p_{i,j} = \hat{y}_{i,j} - \log \left( \sum_{k=1}^{C} e^{\hat{y}_{i,k}} \right)
$$

Ce qui donne:

$$
\mathrm{NLL}(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^{N} \left( \hat{y}_{i, y_i} - \log \sum_{k=1}^{C} e^{\hat{y}_{i,k}} \right)
$$


En pratique, on utilise rarement `NLLLoss` directement **`CrossEntropyLoss` combine déjà `LogSoftmax` et`NLLLoss`** pour une meilleur stabilité numérique.

### 6 - Huber Loss (`nn.SmoothL1Loss`)

Utilisée pour les tâches de **régression** avec à la fois des petites et larges erreurs.
C'est un compromis entre **MSE** qui est sensible aux valeurs abérrantes et **MAE** qui est robuste mais n'obtient pas un résultat lisse.

$$
L_{\delta}(y, \hat{y}) =
\begin{cases}
\frac{1}{2} (y - \hat{y})^2, & \text{if } |y - \hat{y}| \le \delta, \\
\delta \cdot \bigl(|y - \hat{y}| - \tfrac{1}{2}\delta \bigr), & \text{otherwise.}
\end{cases}
$$

Le paramètre $\delta$ défini le point de transition entre la perte quadratique et la perte linéaire.
Cette valeur se trouve expérimentallement afin d'obtenir les meilleurs performances avec notre base de données.

**Huber Loss** est un bon choix pour les utilisations généralistes lorsque l'on recherche à la fois de la stabilité et de la résistance aux valeurs abérrantes dans une régression.

### 7 - KL Divergence Loss (`nn.KLDivLoss`)

Utilisée pour comparer des **distributions de probabilités**, par exemple dans les **Auto-encodeurs variationnels** et les **distillations de connaissances**.

Cela mesure comment une distribution de probabilités $P$ divèrge d'une autre nommée $Q$:

$$
D_{KL}(P || Q) = \sum_{i} P(i) \log \frac{P(i)}{Q(i)}
$$

De façon plus simple, cela nous indique à quel point des informations sont perdues lorsque l'on utilise $Q$ pour approximer $P$.

In [6]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

## Une explication ~~rapide~~ de l’optimiseur

Un **optimiseur** est l’algorithme qui met à jour les paramètres du modèle (poids et biais) à partir des gradients calculés pendant l’entraînement.

Lors d’un passage d’entraînement :

* la passe en avant ou **forward** produit les logits (les prédictions),
* la perte ou **loss** mesure l’erreur,
* le **backward pass** (`loss.backward()`) calcule les gradients,
* puis l’**étape d’optimisation** (`optimizer.step()`) met à jour les poids.

On note la loss par $L$.
La dérivée partielle de $L$ par rapport au paramètre $\theta_i$, notée $\dfrac{\partial L}{\partial \theta_i}$, indique l’ampleur et la direction dans lesquelles la loss varie lorsqu’on modifie $\theta_i$. Cette notion est essentielle pour la suite.


### 1 - La descente de gradient stochastique ou Stochastic Gradient Descent (SGD)

C’est l’optimiseur le plus simple et le plus utilisé historiquement.

**Stochastic** signifie qu’il calcule les mises à jour à partir d’un sous-ensemble aléatoire des données, un *batch*.

Avec toutes les données on appelle ça **Batch Gradient Descent** tandis qu'avec un seul exemple c'est une **Pure SGD**.

En pratique, on utilise des mini-lots ou **Mini-batch SGD**, par exemple de taille 64.

$$
\theta_{t+1} = \theta_t - \eta , \nabla_\theta L(\theta_t)
$$

Avec :

* $\theta_t$ : les paramètres au pas *t*
* $\eta$ : le **taux d'apprentissage** ou **learning rate**
* $\nabla_\theta L(\theta_t)$ : le gradient de la loss par rapport aux paramètres

$$
\nabla_\theta L(\theta_t) =
\begin{bmatrix}
\dfrac{\partial L}{\partial \theta_1} \
\dfrac{\partial L}{\partial \theta_2} \
\vdots \
\dfrac{\partial L}{\partial \theta_n}
\end{bmatrix}
$$

Avec `torch.optim.SGD()`, il suffit de fournir :

* `model.parameters()`
* `lr`

Le taux d'apprentissage détermine la taille des mises à jour :

* `1e-3` → petit pas
* `1e-1` → grand pas

**Limites :**

* même valeur d'$\eta$ pour tous les paramètres
* sensible à l’échelle des gradients
* peut osciller et converger lentement

La SGD seule est surtout utilisée pour les modèles simples.


### 2 - SGD avec Momentum

On améliore SGD en ajoutant un **terme d’inertie** pour stabiliser et accélérer l’apprentissage.

$$
v_t = \beta v_{t-1} + (1 - \beta) , \nabla_\theta L(\theta_t)
$$

$$
\theta_{t+1} = \theta_t - \eta v_t
$$

$v_t$ accumule l’historique des gradients et amortit les oscillations.

On prend généralement $\beta = 0.9$.

**Limites :**

* taux d’apprentissage toujours global
* dépendance forte au choix de $\eta$ et $\beta$

**Usage :**

* très courant pour les **CNN**
* mises à jour stables
* peu coûteux en mémoire
* fonctionne bien avec les planificateurs de taux d'apprentissage


### 3 - Adagrad

$$
G_t = G_{t-1} + (\nabla_\theta L(\theta_t))^2
$$

$$
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \varepsilon}} , \nabla_\theta L(\theta_t)
$$

$G_t$ additionne les gradients au carré.
Chaque paramètre possède son propre $G_t$, donc son **taux d'apprentissage effectif**.

Comme $G_t$ augmente continuellement, le taux d'apprentissage diminue avec le temps.

Adapté aux gradients **dispersés**, par exemple en **NLP**.

**Limite :**
le taux d'apprentissage peut devenir trop faible, bloquant l’apprentissage.


### 4 - RMSProp

RMSProp corrige le problème d’Adagrad en utilisant une **moyenne exponentiellement décroissante** des gradients au carré.

$$
v_t = \alpha v_{t-1} + (1 - \alpha) , (\nabla_\theta L(\theta_t))^2
$$

$$
\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{v_t + \varepsilon}} , \nabla_\theta L(\theta_t)
$$

Avec $\alpha \approx 0.9$.
Les valeurs anciennes dominent (90 %), mais on ajoute 10 % du nouveau gradient au carré.

Très utilisé pour les **RNN**, où les gradients varient beaucoup.


### 5 - Adam

Adam combine les idées du Momentum et de RMSProp :

* moyenne des gradients : $m_t$ (premier moment)
* moyenne des gradients au carré : $v_t$ (second moment)

$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) , \nabla_\theta L(\theta_t)
$$

$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) , (\nabla_\theta L(\theta_t))^2
$$

Ces quantités sont corrigées du biais initial :

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

Mise à jour :

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon}
$$

Adam adapte le taux d'apprentissage par paramètre et lisse les mises à jour.
C’est l’optimiseur **par défaut** dans la majorité des architectures modernes.



### 6 - AdamW (Adam avec une méthode de dégradation des pondérations découplée)

Adam couplait historiquement la méthode de dégradation des pondérations (ou *weight decay*) avec la mise à jour du gradient, ce qui posait problème pour les grands modèles.
AdamW sépare les deux mécanismes.

$$
\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon} + \lambda \theta_t \right)
$$

Seule l’étape finale change :
le terme $\lambda \theta_t$ applique une décroissance des poids indépendante du reste.

Avec $\lambda \approx 10^{-2}$.

AdamW améliore nettement la généralisation, notamment dans les **Transformers** et les grands modèles de vision.

In [7]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Calcul l'erreur de prédiction
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

Dans le *dataloader*, la taille de lot est fixée à 64 : on utilise donc du **Mini-batch SGD**.

Dans la fonction `train(dataloader, model, loss_fn, optimizer)`, les étapes sont les suivantes :

* **`size = len(dataloader.dataset)`**
  Nombre total d’exemples contenus dans le jeu de données.

* **`model.train()`**
  Passage explicite du modèle en **mode entraînement**.
  Certaines couches comme `Dropout` ou `BatchNorm` modifient leur comportement entre l’entraînement et l’évaluation, donc cette instruction est indispensable.

* **`for batch, (X, y) in enumerate(dataloader):`**
  Parcours séquentiel des lots de données.
  `batch` est l’indice du lot.
  `X` regroupe les entrées (par ex. images : $[N, C, H, W]$) et `y` les étiquettes correspondantes ($[N]$ en classification).

* **`X, y = X.to(device), y.to(device)`**
  Envoi des données sur le même périphérique que le modèle (CPU ou GPU).
  Obligatoire lorsque le modèle est sur GPU.

* **`pred = model(X)`**
  Exécution du **forward pass**.
  Cette ligne appelle la méthode `forward` et renvoie les sorties brutes du réseau (**logits** en classification).

* **`loss = loss_fn(pred, y)`**
  Calcul de la **loss**, c’est-à-dire l’écart entre les prédictions et les étiquettes.
  Avec la cross-entropy, on compare les distributions prédites aux classes réelles.
  Le résultat est un scalaire.

* **`loss.backward()`**
  Déclenche la différentiation automatique et la **rétropropagation**.
  PyTorch calcule pour chaque paramètre :

  $$
  \frac{\partial L}{\partial \theta_i}
  $$

  Les gradients sont stockés dans l’attribut `.grad` des paramètres où `requires_grad=True`.

* **`optimizer.step()`**
  Utilisation des gradients pour **mettre à jour les paramètres** selon la règle de l’optimiseur choisi (SGD, Adam, etc.).

* **`optimizer.zero_grad()`**
  Remise à zéro des gradients stockés.
  Cette étape empêche l’**accumulation des gradients** sur plusieurs passes, qui fausserait les mises à jour.

Enfin, le code affiche simplement la **loss tous les 100 lots**, ce qui permet de suivre l’évolution de l’entraînement.

In [10]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Erreur de test: \n Précision: {(100*correct):>0.1f}%, Perte moyenne: {test_loss:>8f} \n")

La fonction `test(dataloader, model, loss_fn)` est une boucle d'évaluation. Elle vérifie comment le modèle performe sur des données qui lui sont inconnues, les données de test.

Il y a certaines parties de la fonction qui pourraient nécéssiter une explication.

* **`model.eval()`**: cela permet de passer au mode d'évaluation, de même que `model.train()` permet de passer au mode d'entraînement. C'est nécessaire s'il y a des couches comme `Dropout` ou `BatchNorm `

* **`with torch.no_grad()`**: ça indique à Pytorch de **ne pas sauvergarder les valeurs du gradient**  pour l'étape de backpropagation. Puisque nous ne faisons que tester notre modèle, nous ne voulons pas l'entraîner sur les données de test, nous n'avons donc pas besoin des données du gradient. Faire cela permet également de réduire l'utilisation de la mémoire RAM et augmente la vitesse de calcul.

* **`test_loss += loss_fn(pred, y).item()`**: ça calcul la fonction de perte de la même façon que dans la fonction d'entraînement.`item()`extrait la valeur scalaire du tenseur.


* **`correct += (pred.argmax(1) == y).type(torch.float).sum().item()`**:

    * `pred.argmax(1)` prend l'indice du plus grand logit de dimension 1, càd la classe prédite, celle qui a le meilleur score.
    *  `== y` compare la prédiction avec la valeur ciblée, renvoyant un tenseur de valeurs booléennes.
    *  `.type(torch.float)` convertis les boléens en des *floats*, des nombres à virgules.
    *  `.sum().item()` somme ces derniers pour obtenir le nombre de prédictions correctes dans ce batch.
    *  `correct +=` accumule le nombre de prédictions correctes à travers tout les batches.


In [11]:
epochs = 20
for t in range(epochs):
    print(f"Époque {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Fini!")

Époque 1
-------------------------------
loss: 0.917870  [   64/60000]
loss: 0.972974  [ 6464/60000]
loss: 0.760904  [12864/60000]
loss: 0.940939  [19264/60000]
loss: 0.817570  [25664/60000]
loss: 0.837432  [32064/60000]
loss: 0.902399  [38464/60000]
loss: 0.854664  [44864/60000]
loss: 0.879731  [51264/60000]
loss: 0.827458  [57664/60000]
Erreur de test: 
 Précision: 69.0%, Perte moyenne: 0.842582 

Époque 2
-------------------------------
loss: 0.854287  [   64/60000]
loss: 0.926266  [ 6464/60000]
loss: 0.704534  [12864/60000]
loss: 0.896237  [19264/60000]
loss: 0.780266  [25664/60000]
loss: 0.790730  [32064/60000]
loss: 0.864123  [38464/60000]
loss: 0.821803  [44864/60000]
loss: 0.838743  [51264/60000]
loss: 0.792331  [57664/60000]
Erreur de test: 
 Précision: 70.5%, Perte moyenne: 0.805459 

Époque 3
-------------------------------
loss: 0.802685  [   64/60000]
loss: 0.887435  [ 6464/60000]
loss: 0.659202  [12864/60000]
loss: 0.860756  [19264/60000]
loss: 0.751029  [25664/60000]
los

Si nécessaire, on peut sauvegarder le modèle avec la commande suivante:

In [12]:
torch.save(model.state_dict(), "modele_du_quickstart.pth")